# 📚 책 데이터를 활용한 추천 시스템 (실습)

Goodreads 데이터를 이용해 **콘텐츠 기반 필터링(Content-based Filtering)** 추천 시스템을 직접 구현합니다.

## 실습 목표
1. 여러 CSV 파일을 불러오고 필요한 정보를 `merge`로 결합할 수 있다.
2. 텍스트 데이터를 **TF-IDF**로 벡터화할 수 있다.
3. **코사인 유사도**(`linear_kernel`)로 아이템 간 유사도 행렬을 만들 수 있다.
4. 유사도 행렬에서 **상위 N개 유사 아이템**을 뽑아 추천 함수를 만들 수 있다.
5. 저자(authors) / 태그(tags) / 저자+태그(corpus) 세 가지 기준의 추천 결과를 비교할 수 있다.

## 사용 데이터
| 파일 | 설명 | 주요 컬럼 |
|---|---|---|
| `books.csv` | 책 메타 정보 (10,000권) | `book_id`, `title`, `authors`, `average_rating` ... |
| `ratings.csv` | 유저의 책 평점 | `book_id`, `user_id`, `rating` |
| `book_tags.csv` | 책에 달린 태그 ID와 횟수 | `goodreads_book_id`, `tag_id`, `count` |
| `tags.csv` | 태그 ID ↔ 태그 이름 | `tag_id`, `tag_name` |
| `to_read.csv` | 유저가 읽고 싶어 한 책 | `user_id`, `book_id` |

## 0. 라이브러리 임포트

In [1]:
# TODO 0-1: pandas, numpy 임포트 + 경고 메시지 무시 설정
import numpy as np, pandas as pd
import warnings; warnings.filterwarnings('ignore')

##### 💡 **힌트**

- 경고 숨기기는 `warnings` 모듈의 `filterwarnings` 함수에 `'ignore'`를 넘기면 됩니다.
- 한 줄에 여러 문장을 쓰고 싶다면 세미콜론(`;`)으로 구분할 수 있습니다.

---

## 1. 데이터 불러오기

CSV 파일들을 DataFrame으로 읽어오고, 각 데이터가 어떤 모양인지 확인합니다.
> 모든 CSV는 노트북과 같은 폴더(`./`)에 있다고 가정합니다.

### TODO 1-1. 책 데이터(`books.csv`)를 불러오세요.
- `books`라는 변수에 저장
- 데이터의 크기(행, 열 개수)를 출력
- 상위 2개 행을 확인

✅ **확인 포인트**: `(10000, 23)` 이 출력되고, `title`, `authors` 등의 컬럼이 보입니다.

In [2]:
# TODO 1-1: books.csv 불러오기 → shape 출력 → head(2) 확인
books = pd.read_csv('./books/books.csv')
print('Shape:', books.shape)
books.head(2).T

Shape: (10000, 23)


,0,1
id,1,2
book_id,2767052,3
best_book_id,2767052,3
work_id,2792775,4640799
books_count,272,491
isbn,439023483,439554934
isbn13,9780439023480.0,9780439554930.0
authors,Suzanne Collins,"J.K. Rowling, Mary GrandPré"
original_publication_year,2008.0,1997.0
original_title,The Hunger Games,Harry Potter and the Philosopher's Stone


##### 💡 **힌트**

### TODO 1-2. 책 태그 데이터(`book_tags.csv`)를 불러오세요.
- `book_tags`라는 변수에 저장하고, shape과 상위 2개 행 확인

✅ **확인 포인트**: `(999912, 3)`

In [3]:
# TODO 1-3: book_tags.csv 불러오기 → shape 출력 → head(2) 확인
book_tags = pd.read_csv('./books/book_tags.csv')
print(book_tags.shape)
book_tags.head(2).T

(999912, 3)


,0,1
goodreads_book_id,1,1
tag_id,30574,11305
count,167697,37174


##### 💡 **힌트**

- 이 데이터에는 태그의 **이름이 없고 `tag_id`만** 있습니다. 이름은 다음 단계에서 붙입니다.
- 책을 가리키는 컬럼 이름이 `book_id`가 아니라 `goodreads_book_id`라는 점을 기억해 두세요. (나중에 merge 할 때 중요합니다.)

### TODO 1-3. 태그 정보(`tags.csv`)를 불러오세요.
- `tags`라는 변수에 저장하고, shape과 **마지막** 2개 행 확인

✅ **확인 포인트**: `(34252, 2)`

In [ ]:
# TODO 1-4: tags.csv 불러오기 → shape 출력 → tail(2) 확인

tags = pd.read_csv('./books/tags.csv')
print(tags.shape)
tags.tail(2).T

# favorites
# ｆａｖｏｕｒｉｔｅｓ
# ascii 코드로 변환하는게 필요할 수 있다.

(34252, 2)


,34250,34251
tag_id,34250,34251
tag_name,ＳＥＲＩＥＳ,ｆａｖｏｕｒｉｔｅｓ


##### 💡 **힌트**

- 마지막 행 확인은 `.tail(n)`을 사용합니다.
- 마지막 부분을 보면 전각 문자로 된 태그 등 **정제되지 않은 태그**가 섞여 있는 것을 볼 수 있습니다.   
    -> 실제 서비스라면 전처리가 필요한 부분입니다.

### TODO 1-4. `book_tags`와 `tags`를 결합해 태그 이름을 붙이세요.
- 두 데이터를 `tag_id` 기준으로 합쳐 `tags_join_df`에 저장
- 상위 2개 행 확인

✅ **확인 포인트**: `goodreads_book_id`, `tag_id`, `count`, **`tag_name`** 4개 컬럼이 생기고, 첫 행의 태그가 `to-read`로 보입니다.

In [6]:
# TODO 1-5: book_tags + tags 를 tag_id 기준으로 merge → tags_join_df

tags_join_df = pd.merge(book_tags, tags, on='tag_id')
tags_join_df.head(2)

,goodreads_book_id,tag_id,count,tag_name
0,1,30574,167697,to-read
1,1,11305,37174,fantasy


##### 💡 **힌트**

- 결합은 `pd.merge(왼쪽DF, 오른쪽DF, left_on=..., right_on=..., how=...)`
- 양쪽 모두 `tag_id`라는 같은 이름의 컬럼을 가지고 있습니다.
- 양쪽에 모두 존재하는 행만 남기려면 `how='inner'`를 사용합니다.

---

## 2. 저자(authors) 기반 추천 — TF-IDF 벡터화

첫 번째 아이디어: **"같은 작가의 책이라면 취향이 비슷할 것이다."**

이를 위해 `authors` 컬럼(텍스트)을 숫자 벡터로 바꿔야 합니다. 이때 사용하는 것이 **TF-IDF**입니다.

> **TF-IDF란?**
> - **TF (Term Frequency)**: 한 문서에서 특정 단어가 얼마나 자주 등장하는가
> - **IDF (Inverse Document Frequency)**: 그 단어가 전체 문서에서 얼마나 희귀한가
> - 두 값을 곱해 "이 문서에서는 자주 나오지만 다른 문서에서는 드문 단어"에 높은 가중치를 줍니다.

### TODO 2-1. `books`의 `authors` 컬럼 상위 5개 값을 확인하세요.

In [ ]:
# TODO 2-1: books의 authors 컬럼 상위 5개 확인
books['authors'].head(5) # books.head(5)['authors']
books['authors'][:5]

0                Suzanne Collins
1    J.K. Rowling, Mary GrandPré
2                Stephenie Meyer
3                     Harper Lee
4            F. Scott Fitzgerald
Name: authors, dtype: str

##### 💡 **힌트**

- 특정 컬럼 선택: `books['컬럼명']`
- 앞의 5개만 보기: 슬라이싱 `[:5]` 또는 `.head()`
- 값을 보면서 **한 셀에 여러 저자가 쉼표로 들어있는 경우**가 있다는 점을 확인하세요. 이 때문에 단순 문자열 비교가 아니라 벡터화가 필요합니다.

### TODO 2-2. `authors`를 TF-IDF로 벡터화하세요.
- `sklearn`에서 TF-IDF 벡터라이저를 임포트
- 아래 조건으로 벡터라이저 객체를 만들어 `tf`에 저장
  - 분석 단위(analyzer): 단어
  - n-gram 범위(ngram_range): 1~2 (unigram + bigram)
  - 최소 문서 빈도(min_df): 0.0
  - 영어 불용어 제거(stop_words)
- `books['authors']`를 학습 + 변환하여 `tfidf_matrix`에 저장하고 결과를 확인

✅ **확인 포인트**: shape이 `(10000, 14800)` 형태의 sparse matrix가 출력됩니다.

In [11]:
# TODO 2-2: TfidfVectorizer 임포트 → tf 객체 생성 → authors 벡터화 → tfidf_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=0.0, stop_words='english')
tfidf_matrix = tf.fit_transform(books['authors'])
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 43199 stored elements and shape (10000, 14800)>

In [ ]:
10000*14800*64/8/1024/1024/1024 # Byte -> KB -> MB -> GB

1.1026859283447266

In [ ]:
2**32 # uint32

# [(uint32, uint32, float64), ...]
# (행 위치, 열 위치, 값)

41999 * 128/8/1024/1024 # MB

0.6408538818359375


##### 💡 **힌트**


- 임포트 경로: `from sklearn.feature_extraction.text import TfidfVectorizer`
- 파라미터 이름: `analyzer`, `ngram_range`, `min_df`, `stop_words`
  - 분석 단위는 문자열 `'word'`, n-gram 범위는 튜플 `(1, 2)`, 불용어는 `'english'`
- 학습과 변환을 한 번에: `.fit_transform(데이터)`
- 결과는 대부분의 값이 0인 **희소 행렬(sparse matrix)** 입니다. 행 개수는 책의 수, 열 개수는 단어(feature)의 수입니다.


---

## 3. 유사도 행렬 만들기 (linear_kernel)

벡터로 바꿨으니 이제 **책과 책 사이가 얼마나 비슷한지** 계산합니다.

> **왜 `linear_kernel`인가?**  
> 코사인 유사도는 `(A·B) / (|A||B|)` 입니다. 그런데 TF-IDF 결과는 이미 **L2 정규화(길이가 1)** 되어 있어서 분모가 1이 됩니다.  
> 즉 **내적(dot product)만 계산하면 = 코사인 유사도**가 되며, `linear_kernel`이 바로 이 내적을 계산합니다. `cosine_similarity`보다 빠릅니다.

### TODO 3-1. TF-IDF 행렬로 코사인 유사도 행렬을 만드세요.
- `sklearn`에서 `linear_kernel`을 임포트
- `tfidf_matrix`를 자기 자신과 비교하여 `cosine_sim`에 저장하고 결과 확인

✅ **확인 포인트**: shape이 `(10000, 10000)`이고 대각 성분이 1인 배열이 출력됩니다.

In [22]:
# TODO 3-1: linear_kernel 임포트 → cosine_sim 계산
from sklearn.metrics.pairwise import linear_kernel

cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(10000, 10000))

##### 💡 **힌트**

- 임포트 경로: `from sklearn.metrics.pairwise import linear_kernel`
- 모든 책 쌍을 비교해야 하므로 **같은 행렬을 두 번** 인자로 넘깁니다.
- 결과는 (책 수 × 책 수) 정사각 행렬이고, **대각선은 자기 자신과의 유사도라 항상 1**입니다.

---

## 4. 저자 기준으로 유사한 책 찾기 (단계별)

유사도 행렬이 있어도, 사용자는 **책 제목**으로 검색합니다.  
따라서 `제목 → 행 인덱스`로 바꿔주는 **매핑**이 필요합니다.

### TODO 4-1. 제목으로 인덱스를 찾을 수 있는 매핑을 만드세요.
- 책 제목만 담은 Series를 `titles`에 저장
- **값이 인덱스 번호, 인덱스가 책 제목**인 Series를 만들어 `indices`에 저장
- `'The Hobbit'`의 인덱스를 조회해 보세요

✅ **확인 포인트**: `6`이 반환됩니다.

In [27]:
# TODO 4-1: titles, indices 생성 → indices['The Hobbit'] 조회
titles = books['title']
indices = pd.Series(titles.index, index=titles)
indices['The Hobbit']

np.int64(6)

##### 💡 **힌트**

- Series 생성: `pd.Series(데이터, index=인덱스로_쓸_값)`
- 여기서 "데이터"는 DataFrame의 행 번호인 `books.index`, "인덱스"는 `books['title']`입니다.
- 조회는 딕셔너리처럼 `indices['책제목']`

### TODO 4-2. `'The Hobbit'`과 다른 모든 책 사이의 유사도 값을 꺼내 보세요.

✅ **확인 포인트**: 길이 10000짜리 1차원 배열이 출력됩니다.

In [34]:
# TODO 4-2: 'The Hobbit'의 유사도 벡터 확인
cosine_sim[ indices['The Hobbit'] ]

array([0., 0., 0., ..., 0., 0., 0.], shape=(10000,))

##### 💡 **힌트**

- `cosine_sim`은 2차원 배열입니다. `cosine_sim[행번호]`를 하면 그 책과 **전체 책** 간의 유사도 1차원 배열이 나옵니다.
- 행 번호 자리에 4-1에서 만든 `indices`를 활용하세요.

### TODO 4-3. 유사도가 높은 순서로 정렬하세요.
- 유사도 배열을 `(인덱스, 유사도)` 쌍의 리스트로 변환
- 유사도 기준 **내림차순** 정렬
- 상위 3개를 확인

✅ **확인 포인트**: 첫 번째 원소가 `(6, 1.0)` — 즉 **자기 자신**입니다. (다음 단계에서 이걸 빼야 합니다!)

In [ ]:
# TODO 4-3: 4-2에서 구한 유사도 벡터를 (인덱스, 유사도) 리스트로 만들고 유사도 내림차순 정렬 → sim_scores
sim_scores = list(enumerate( cosine_sim[ indices['The Hobbit'] ] ))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
sim_scores[:3]

[(6, np.float64(1.0)), (18, np.float64(1.0)), (154, np.float64(1.0))]

##### 💡 **힌트**

- 값에 순번을 붙이려면 `enumerate()`를 쓰고, `list()`로 감싸면 튜플 리스트가 됩니다.
- 정렬: `sorted(리스트, key=..., reverse=True)`
- `key`에는 **튜플의 두 번째 원소(유사도)**를 반환하는 `lambda x: x[1]`을 넘깁니다.
- `reverse=True`가 내림차순입니다.

### TODO 4-4. 자기 자신을 제외한 상위 10권의 제목을 출력하세요.
- 정렬된 결과에서 **1번째부터 10번째까지** 잘라내기
- 각 튜플에서 **인덱스만** 추출해 리스트로 만들기
- 그 인덱스로 `titles`에서 책 제목 가져오기

✅ **확인 포인트**: 반지의 제왕 시리즈, 실마릴리온 등 **톨킨의 다른 작품들**이 나옵니다. 저자 기반 추천이 잘 동작한 것입니다.

In [ ]:
# TODO 4-4: 상위 10권(자기 자신 제외)의 제목 출력
top10 = sim_scores[1:11]
book_idxs = [i for i, _ in top10]
titles[book_idxs]

18      The Fellowship of the Ring (The Lord of the Ri...
154            The Two Towers (The Lord of the Rings, #2)
160     The Return of the King (The Lord of the Rings,...
188     The Lord of the Rings (The Lord of the Rings, ...
963     J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...
4975         Unfinished Tales of Númenor and Middle-Earth
2308                                The Children of Húrin
610              The Silmarillion (Middle-Earth Universe)
8271                   The Complete Guide to Middle-Earth
1128     The History of the Hobbit, Part One: Mr. Baggins
Name: title, dtype: str

##### 💡 **힌트**

- 자기 자신은 항상 맨 앞(0번)에 오므로 슬라이싱을 `[1:11]`로 합니다.
- 인덱스만 뽑기: 리스트 컴프리헨션 `[i[0] for i in ...]`
- 위치 기반으로 여러 행 선택: `titles.iloc[인덱스리스트]`
  - ⚠️ `loc`이 아니라 **`iloc`** 입니다. 우리가 가진 것은 라벨이 아니라 **행 위치**이기 때문입니다.

---

## 5. 태그(tag) 기반 추천

두 번째 아이디어: **"같은 태그가 붙은 책이라면 분위기/장르가 비슷할 것이다."**

저자 기반은 같은 작가의 책만 추천하는 한계가 있습니다. 태그를 쓰면 **다른 작가의 비슷한 책**도 찾을 수 있습니다.

### TODO 5-1. `books`와 `tags_join_df`를 결합하세요.
- 결과를 `books_with_tags`에 저장하고 상위 2개 행 확인

✅ **확인 포인트**: 컬럼이 27개가 되고, `tag_name` 컬럼이 추가되어 있습니다.

In [ ]:
# book_id / goodreads_book_id
books

In [42]:
# TODO 5-1: books + tags_join_df merge → books_with_tags
books_with_tags = pd.merge(books, tags_join_df, 
                           left_on='book_id', 
                           right_on='goodreads_book_id')
books_with_tags.head(2)

,id,book_id,best_book_id,work_id,books_count,isbn,isbn13,authors,original_publication_year,original_title,...,ratings_2,ratings_3,ratings_4,ratings_5,image_url,small_image_url,goodreads_book_id,tag_id,count,tag_name
0,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,2767052,11557,50755,favorites
1,1,2767052,2767052,2792775,272,439023483,9.780439e+12,Suzanne Collins,2008.0,The Hunger Games,...,127936,560092,1481305,2706317,https://images.gr-assets.com/books/1447303603m...,https://images.gr-assets.com/books/1447303603s...,2767052,8717,35418,currently-reading


##### 💡 **힌트**

- 두 데이터에서 책을 가리키는 컬럼 이름이 **서로 다릅니다**: `books`는 `book_id`, `tags_join_df`는 `goodreads_book_id`
- 이럴 때는 `pd.merge(..., left_on=..., right_on=..., how='inner')`로 각각 지정합니다.
- 한 책에 태그가 여러 개 달려 있으므로, merge 후 **같은 책이 여러 행으로 중복**됩니다. 정상입니다.

### TODO 5-2. 태그 이름을 TF-IDF로 벡터화하고 유사도 행렬을 만드세요.
- 2-2와 **같은 옵션**으로 벡터라이저를 만들어 `tf1`에 저장
- `books_with_tags['tag_name']`의 **상위 10000개 행**만 사용해 `tfidf_matrix1` 생성
- `linear_kernel`로 `cosine_sim1` 생성

✅ **확인 포인트**: `tfidf_matrix1`의 shape이 `(10000, 1381)`, `cosine_sim1`이 `(10000, 10000)`

In [46]:
# TODO 5-2: tag_name 기준 TF-IDF (tf1, tfidf_matrix1) → cosine_sim1
tf1 = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=0.0, stop_words='english')
tfidf_matrix1 = tf1.fit_transform(books_with_tags['tag_name'].head(10000))
tfidf_matrix1

cosine_sim1 = linear_kernel(tfidf_matrix1, tfidf_matrix1)
cosine_sim1

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(10000, 10000))

##### 💡 **힌트**

- 앞부분 n개만 쓰려면 `.head(10000)`
- 나머지 과정은 2-2, 3-1과 완전히 동일합니다.
- ⚠️ 여기서 `.head(10000)`은 **책 10000권**이 아니라 **merge된 행 10000개**입니다. 즉 실제로는 몇백 권 분량의 태그만 담깁니다.  
  → 이 불일치가 다음 단계에서 이상한 추천 결과를 만듭니다. 왜 그런지 생각하며 진행해 보세요.

### TODO 5-3. 태그 기준 추천 함수 `tags_recommendations`를 작성하세요.

**함수 명세**
- 이름: `tags_recommendations`
- 인자: `title`(책 제목), `top_n`(추천 개수, 기본값 10)
- 반환: 추천된 책 제목 Series
- 함수 정의 후 `'The Hobbit'`에 대해 15개를 추천받아 보세요.

In [ ]:
# TODO 5-3: tags_recommendations 함수 정의 → 'The Hobbit' 15개 추천 
# (앞에서 만든 titles, indices 활용)

def tags_recommendations(title, top_n=10):
    idx = indices[title]
    sim_scores = list(enumerate( cosine_sim1[ idx ] ))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    topn = sim_scores[1:top_n+1]
    book_idxs = [i for i, _ in topn]

    return titles[book_idxs]

tags_recommendations('The Hobbit', 15)

1196    Free Four: Tobias Tells the Divergent Knife-Th...
1299                                    Those Who Save Us
1304        The Boxcar Children (The Boxcar Children, #1)
1601                     Changes (The Dresden Files, #12)
1905          Rules of Attraction (Perfect Chemistry, #2)
2710                          Incarceron (Incarceron, #1)
3513     First Drop of Crimson (Night Huntress World, #1)
4707                 BookRags Summary:  A Storm of Swords
5406                                     Telegraph Avenue
6471                     The Hidden City (The Tamuli, #3)
6805                     The Honk and Holler Opening Soon
6916                                          Absurdistan
7206                                      I Suck at Girls
9005                         The Education of Little Tree
1194                                      The Round House
Name: title, dtype: str

🤔 **생각해 볼 점**: 결과가 톨킨/판타지와 전혀 관계없는 책들로 나옵니다. **왜 그럴까요?**
> `cosine_sim1`의 행 번호는 `books_with_tags`(merge로 중복된 데이터)의 행 번호인데, `indices1`은 `books`의 행 번호를 가리킵니다.  
    **두 인덱스 체계가 서로 어긋나 있는 것**이 원인입니다. 아래에서 문제를 해결해봅시다.

##### 💡 **힌트**

- 4-1 ~ 4-4에서 한 과정을 **그대로 함수 안에 묶는 것**입니다. 순서는 다음과 같습니다.
  1. `title`로 인덱스 찾기
  2. `cosine_sim1[인덱스]`를 `enumerate` + `list`로 변환
  3. 유사도 내림차순 정렬
  4. `[1:top_n+1]`로 자기 자신 제외하고 top_n개 자르기
  5. 인덱스만 추출
  6. `.iloc[]`으로 제목 반환
- 함수 밖에 `titles1`, `indices1`을 미리 만들어 두고 함수 안에서 참조하면 됩니다.
- 기본값이 있는 인자: `def 함수명(title, top_n=10):`

---

## 6. corpus(저자 + 태그) 기반 추천

앞의 문제를 해결하면서 두 정보를 모두 쓰는 방법:  
**책 한 권당 태그들을 하나의 문자열로 합친 뒤**, 저자 이름과 이어 붙여 하나의 **corpus(말뭉치)** 를 만듭니다.

이렇게 하면 한 책 = 한 행이 되어 인덱스 어긋남 문제가 사라지고, 저자와 장르 정보를 동시에 반영할 수 있습니다.

### TODO 6-1. 책별로 태그들을 하나의 문자열로 합치세요.
- `books_with_tags`를 책 단위로 묶어 `tag_name`을 **공백으로 연결**
- 결과를 `temp_df`에 저장하고 상위 2개 행 확인

✅ **확인 포인트**: `book_id`와 `tag_name` 2개 컬럼이 있고, `tag_name` 값이 `"to-read fantasy favorites ..."` 형태의 긴 문자열입니다.

In [50]:
# TODO 6-1: books_with_tags를 book_id로 groupby → tag_name을 공백으로 join → temp_df
temp_df = books_with_tags.groupby('book_id')['tag_name'].apply(' '.join).reset_index()
temp_df.head(2)

,book_id,tag_name
0,1,to-read fantasy favorites currently-reading yo...
1,2,to-read currently-reading fantasy favorites ch...


##### 💡 **힌트**

- 그룹화: `.groupby('묶을컬럼')['대상컬럼']`
- 그룹별로 함수 적용: `.apply(함수)`
- 문자열 리스트를 공백으로 잇기: `' '.join` (괄호 없이 함수 자체를 넘깁니다)
- `groupby` 결과는 그룹 키가 인덱스가 되므로, 일반 컬럼으로 되돌리려면 `.reset_index()`

### TODO 6-2. `temp_df`를 `books`에 결합하세요.
- `book_id` 기준으로 합쳐 다시 `books`에 저장하고 상위 2개 행 확인

✅ **확인 포인트**: `books`의 컬럼 수가 24개가 되고 맨 뒤에 `tag_name`이 붙습니다.

In [ ]:
# 혹시 두번 실행했거나 하는 경우 tag_name_?? 형태로 칼럼에 들어감
# books.drop(['tag_name_x', 'tag_name_y'], axis=1, inplace=True)

In [ ]:
# TODO 6-2: books와 temp_df를 book_id 기준으로 merge → books
books = pd.merge(books, temp_df, on='book_id')
books

##### 💡 **힌트**

- 이번엔 양쪽 키 컬럼 이름이 `book_id`로 동일합니다.
- `how='inner'`를 사용합니다.
- ⚠️ 이 셀을 **두 번 이상 실행하면** `tag_name_x`, `tag_name_y`처럼 컬럼이 중복 생성될 수 있습니다. 그럴 땐 커널을 재시작하고 처음부터 다시 실행하세요.

### TODO 6-3. 저자 이름과 태그를 합쳐 `corpus` 컬럼을 만드세요.
- `authors`와 `tag_name` 두 컬럼을 공백으로 이어 붙여 `books['corpus']`에 저장
- 상위 3개 값 확인

✅ **확인 포인트**: `"Suzanne Collins favorites currently-reading young-adult ..."` 처럼 저자 뒤에 태그가 이어집니다.

In [63]:
# TODO 6-3: authors + tag_name을 합쳐 books['corpus'] 생성
# books['corpus'] = books['authors'] + ' ' + books['tag_name']
books['corpus'] = pd.Series(
    books[['authors', 'tag_name']].fillna('').values.tolist()).str.join(' ')


In [64]:
books['corpus']

0       Suzanne Collins favorites currently-reading yo...
1       J.K. Rowling, Mary GrandPré to-read favorites ...
2       Stephenie Meyer young-adult fantasy favorites ...
3       Harper Lee classics favorites to-read classic ...
4       F. Scott Fitzgerald classics favorites fiction...
                              ...                        
9995    Ilona Andrews to-read urban-fantasy fantasy ro...
9996    Robert A. Caro to-read biography history polit...
9997    Patrick O'Brian to-read historical-fiction fic...
9998    Peggy Orenstein to-read non-fiction nonfiction...
9999    John Keegan to-read history currently-reading ...
Name: corpus, Length: 10000, dtype: object

##### 💡 **힌트**

- 간단하게 `books['authors'] + ' ' + books['tag_name']`로 작성할 수 있습니다.

### TODO 6-4. corpus로 TF-IDF와 유사도 행렬을 만들고, 매핑을 갱신하세요.
- 같은 옵션의 벡터라이저를 `tf_corpus`에 생성
- `books['corpus']`를 벡터화해 `tfidf_matrix_corpus` 생성
- `linear_kernel`로 `cosine_sim_corpus` 생성
- `titles`, `indices`를 **merge된 최신 `books` 기준으로 다시** 생성

In [65]:
# TODO 6-4: corpus 기준 TF-IDF → cosine_sim_corpus → titles, indices 재생성
tf_corpus = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=0.0, stop_words='english')
tfidf_matrix_corpus = tf_corpus.fit_transform(books['corpus'])
tfidf_matrix_corpus

cosine_sim_corpus = linear_kernel(tfidf_matrix_corpus, tfidf_matrix_corpus)
cosine_sim_corpus

titles = books['title']
indices = pd.Series(titles.index, index=titles)

In [67]:
cosine_sim_corpus

array([[1.        , 0.136213  , 0.14082287, ..., 0.03670601, 0.05440203,
        0.01269601],
       [0.136213  , 1.        , 0.1353014 , ..., 0.0353029 , 0.05259607,
        0.01852855],
       [0.14082287, 0.1353014 , 1.        , ..., 0.02233783, 0.03126792,
        0.0135636 ],
       ...,
       [0.03670601, 0.0353029 , 0.02233783, ..., 1.        , 0.02141224,
        0.0677429 ],
       [0.05440203, 0.05259607, 0.03126792, ..., 0.02141224, 1.        ,
        0.05860747],
       [0.01269601, 0.01852855, 0.0135636 , ..., 0.0677429 , 0.05860747,
        1.        ]], shape=(10000, 10000))

##### 💡 **힌트**

- 앞선 과정(2-2, 3-1, 4-1)과 동일합니다.
- ⚠️ `titles`, `indices`를 **반드시 다시 만들어야** 합니다.  
    6-2의 merge로 `books`의 행 구성이 바뀌었기 때문에, 예전 매핑을 그대로 쓰면 5-3과 같은 인덱스 어긋남 문제가 다시 발생합니다.

### TODO 6-5. corpus 기준 추천 함수 `corpus_recommendations`를 작성하세요.

**함수 명세**
- 이름: `corpus_recommendations`
- 인자: `title`, `top_n=10`
- 반환: 추천된 책 제목 Series

In [ ]:
# TODO 6-5: corpus_recommendations 함수 정의
def corpus_recommendations(title, top_n=10):
    idx = indices[title]
    sim_scores = list(enumerate( cosine_sim_corpus[ idx ] ))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    topn = sim_scores[1:top_n+1]
    book_idxs = [i for i, _ in topn]

    return titles[book_idxs]

##### 💡 **힌트**

- 5-3의 `tags_recommendations`와 구조가 **완전히 동일**합니다. 참조하는 유사도 행렬(`cosine_sim_corpus`)과 매핑(`indices`, `titles`)만 바꾸면 됩니다.

### TODO 6-6. `'The Hobbit'`으로 15권을 추천받아 보세요.

✅ **확인 포인트**: 반지의 제왕 시리즈뿐 아니라 `The Once and Future King`, `A Wizard of Earthsea`, `The Golden Compass` 등 **다른 작가의 판타지 작품**까지 추천됩니다. 저자 기반(4-4)과 비교해 보세요.

In [70]:
# TODO 6-6: corpus_recommendations로 'The Hobbit' 15권 추천
corpus_recommendations('The Hobbit', 15)

188     The Lord of the Rings (The Lord of the Rings, ...
154            The Two Towers (The Lord of the Rings, #2)
160     The Return of the King (The Lord of the Rings,...
610              The Silmarillion (Middle-Earth Universe)
4975         Unfinished Tales of Númenor and Middle-Earth
18      The Fellowship of the Ring (The Lord of the Ri...
2308                                The Children of Húrin
8271                   The Complete Guide to Middle-Earth
963     J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...
465                             The Hobbit: Graphic Novel
1128     The History of the Hobbit, Part One: Mr. Baggins
1366    The Once and Future King (The Once and Future ...
479           The Amber Spyglass (His Dark Materials, #3)
592             A Wizard of Earthsea (Earthsea Cycle, #1)
61            The Golden Compass (His Dark Materials, #1)
Name: title, dtype: str

### TODO 6-7. `'Twilight (Twilight, #1)'`로도 15권을 추천받아 보세요.

✅ **확인 포인트**: 트와일라잇 시리즈 전권과 그래픽 노블, 그리고 같은 작가의 `The Host`까지 추천됩니다.

In [69]:
# TODO 6-7: corpus_recommendations로 'Twilight (Twilight, #1)' 15권 추천
corpus_recommendations('Twilight (Twilight, #1)', 15)

51                                 Eclipse (Twilight, #3)
48                                New Moon (Twilight, #2)
991                    The Twilight Saga (Twilight, #1-4)
833                         Midnight Sun (Twilight, #1.5)
731     The Short Second Life of Bree Tanner: An Eclip...
1618    The Twilight Saga Complete Collection  (Twilig...
2020             The Twilight Collection (Twilight, #1-3)
4087    The Twilight Saga: The Official Illustrated Gu...
219     Twilight: The Complete Illustrated Movie Compa...
72                                The Host (The Host, #1)
55                           Breaking Dawn (Twilight, #4)
3074    Twilight: The Graphic Novel, Vol. 1 (Twilight:...
5244    Twilight: The Graphic Novel, Vol. 2  (Twilight...
635     The Twilight Saga Breaking Dawn Part 1: The Of...
953     Twilight Director's Notebook : The Story of Ho...
Name: title, dtype: str

---

## 🎉 수고하셨습니다!

### 핵심 정리
| 단계 | 사용 기술 | 배운 것 |
|---|---|---|
| 데이터 결합 | `pd.merge`, `groupby().apply()` | 서로 다른 키 이름 처리, 다:1 관계를 문자열로 집계 |
| 텍스트 → 벡터 | `TfidfVectorizer` | 희귀하면서 문서를 잘 대표하는 단어에 가중치 부여 |
| 유사도 계산 | `linear_kernel` | L2 정규화된 TF-IDF에서는 내적 = 코사인 유사도 |
| 추천 생성 | `enumerate` + `sorted` + `iloc` | 자기 자신 제외, 상위 N개 선택 |
| **가장 중요** | 인덱스 정합성 | 유사도 행렬의 행 번호와 매핑의 인덱스 체계가 반드시 일치해야 함 |